<a href="https://colab.research.google.com/github/anuvishalp/Python_Projects/blob/main/FlaskApi-%20FastApi%20Implementation-SwaggerUI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Python exercies

1. Implement a RESTful API for tracking IOUs.

        Four roommates have a habit of borrowing money from each other frequently, and have trouble remembering who owes whom, and how much.

        Your task is to implement a simple RESTful API that receives IOUs as POST requests, and can deliver specified summary information via GET requests.

        API Specification
        User object
        {
          "name": "Adam",
          "owes": {
            "Bob": 12.0,
            "Chuck": 4.0,
            "Dan": 9.5
          },
          "owed_by": {
            "Bob": 6.5,
            "Dan": 2.75
          },
          "balance": "<(total owed by other users) - (total owed to other users)>"
        }

Methods
          Description	HTTP Method	URL	Payload Format	Response w/o Payload	Response w/ Payload
          List of user information	GET	/users	{"users":["Adam","Bob"]}	{"users":<List of all User objects>}	{"users":<List of User objects for <users> (sorted by name)}
          Create user	POST	/add	{"user":<name of new user (unique)>}	N/A	<User object for new user>
          Create IOU	POST	/iou	{"lender":<name of lender>,"borrower":<name of borrower>,"amount":5.25}
          Here’s the complete, clean, production‑ready implementation of the IOU‑tracking REST API you described — fully aligned with the spec, deterministic, and easy to extend.


------------>Full Flask implementation (copy‑paste ready)

🚀 Core Idea
Each user has:
      owes → money they owe others
      owed_by → money others owe them
      balance = sum(owed_by) − sum(owes)
IOUs update both users symmetrically.

🧱 Data Model (in‑memory)
          users = {}
Each user stored as:
          {
              "name": "Adam",
              "owes": {},
              "owed_by": {},
              "balance": 0.0
          }
================================================
🧩 Full Flask Implementation

from flask import Flask, request, jsonify
app = Flask(__name__)
users = {}
def calculate_balance(user):
    return sum(user["owed_by"].values()) - sum(user["owes"].values())

def get_user(name):
    return users.get(name)

def create_user(name):
    user = {
        "name": name,
        "owes": {},
        "owed_by": {},
        "balance": 0.0
    }
    users[name] = user
    return user

def update_iou(lender, borrower, amount):
    lender_user = get_user(lender)
    borrower_user = get_user(borrower)

    # Borrower owes lender
    borrower_user["owes"][lender] = borrower_user["owes"].get(lender, 0) + amount
    lender_user["owed_by"][borrower] = lender_user["owed_by"].get(borrower, 0) + amount

    # Netting out if both owe each other
    if lender in borrower_user["owed_by"]:
        cross = min(borrower_user["owed_by"][lender], borrower_user["owes"][lender])
        borrower_user["owed_by"][lender] -= cross
        borrower_user["owes"][lender] -= cross
        if borrower_user["owed_by"][lender] == 0:
            del borrower_user["owed_by"][lender]
        if borrower_user["owes"][lender] == 0:
            del borrower_user["owes"][lender]

    if borrower in lender_user["owes"]:
        cross = min(lender_user["owes"][borrower], lender_user["owed_by"][borrower])
        lender_user["owed_by"][borrower] -= cross
        lender_user["owes"][borrower] -= cross
        if lender_user["owed_by"][borrower] == 0:
            del lender_user["owed_by"][borrower]
        if lender_user["owes"][borrower] == 0:
            del lender_user["owes"][borrower]

    # Recalculate balances
    lender_user["balance"] = calculate_balance(lender_user)
    borrower_user["balance"] = calculate_balance(borrower_user)

    return [lender_user, borrower_user]


@app.route("/users", methods=["GET"])
def list_users():
    req = request.get_json(silent=True)
    if req and "users" in req:
        selected = sorted([users[name] for name in req["users"]], key=lambda u: u["name"])
        return jsonify({"users": selected})
    return jsonify({"users": sorted(users.values(), key=lambda u: u["name"])})


@app.route("/add", methods=["POST"])
def add_user():
    name = request.json["user"]
    user = create_user(name)
    return jsonify(user)


@app.route("/iou", methods=["POST"])
def create_iou():
    lender = request.json["lender"]
    borrower = request.json["borrower"]
    amount = request.json["amount"]

    updated = update_iou(lender, borrower, amount)
    updated_sorted = sorted(updated, key=lambda u: u["name"])
    return jsonify({"users": updated_sorted})


if __name__ == "__main__":
    app.run(debug=True)
================================================
🚀 API Specification

📌 Behavior Notes

          1. Balance calculation -> Always recomputed after each IOU.
          2. Netting out cross‑debts ->If A owes B and B owes A, the smaller amount cancels.
          3. Sorted output ->All user lists must be sorted alphabetically.

📬 Example Requests

Create users
        POST /add
        {"user": "Adam"}

Create IOU
      POST /iou
      {"lender": "Adam", "borrower": "Bob", "amount": 5.25}

List all users
      GET /users
List specific users
       GET /users
      {"users": ["Bob", "Adam"]}

In [ ]:
### Implementation of FlaskAPi -> step by step

⭐ Step 1 — Create a project folder

On your computer, create a folder named -> iou_flask_api
Open VS Code ->
      Click File → Open Folder
      Select iou_flask_api
====================================
⭐ Step 2 — Open the VS Code Terminal
Inside VS Code:  Press Ctrl + ` (backtick)
                 A terminal opens at the bottom

You will run all commands here.
====================================
⭐ Step 3 — Create a virtual environment
In the terminal:  -> python -m venv .venv
Activate it:

        Windows (PowerShell)
                bash
                .venv\Scripts\Activate
        macOS / Linux
                bash
                source .venv/bin/activate
You should now see:

Code
(.venv)
at the beginning of your terminal line.
====================================
⭐ Step 4 — Install Flask -> pip install flask
⭐ Step 5 — Create your Flask API file
In VS Code:  Right‑click the folder → New File
            Name it: "app.py"

app.py -> Paste this working IOU API:

                from flask import Flask, request, jsonify

                app = Flask(__name__)

                # In-memory "database"
                users = {}


                def calculate_balance(user):
                    return sum(user["owed_by"].values()) - sum(user["owes"].values())


                def create_user(name):
                    user = {
                        "name": name,
                        "owes": {},
                        "owed_by": {},
                        "balance": 0.0,
                    }
                    users[name] = user
                    return user


                def update_iou(lender, borrower, amount):
                    lender_user = users[lender]
                    borrower_user = users[borrower]

                    # Borrower owes lender
                    borrower_user["owes"][lender] = borrower_user["owes"].get(lender, 0.0) + amount
                    lender_user["owed_by"][borrower] = lender_user["owed_by"].get(borrower, 0.0) + amount

                    # Net out cross-debts on borrower side
                    if lender in borrower_user["owed_by"]:
                        cross = min(borrower_user["owed_by"][lender], borrower_user["owes"][lender])
                        borrower_user["owed_by"][lender] -= cross
                        borrower_user["owes"][lender] -= cross
                        if borrower_user["owed_by"][lender] == 0:
                            del borrower_user["owed_by"][lender]
                        if borrower_user["owes"][lender] == 0:
                            del borrower_user["owes"][lender]

                    # Net out cross-debts on lender side
                    if borrower in lender_user["owes"]:
                        cross = min(lender_user["owes"][borrower], lender_user["owed_by"][borrower])
                        lender_user["owed_by"][borrower] -= cross
                        lender_user["owes"][borrower] -= cross
                        if lender_user["owed_by"][borrower] == 0:
                            del lender_user["owed_by"][borrower]
                        if lender_user["owes"][borrower] == 0:
                            del lender_user["owes"][borrower]

                    # Recalculate balances
                    lender_user["balance"] = calculate_balance(lender_user)
                    borrower_user["balance"] = calculate_balance(borrower_user)

                    return [lender_user, borrower_user]


                @app.route("/users", methods=["GET"])
                def list_users():
                    req = request.get_json(silent=True)
                    if req and "users" in req:
                        selected = sorted(
                            [users[name] for name in req["users"]],
                            key=lambda u: u["name"],
                        )
                        return jsonify({"users": selected})
                    # all users
                    return jsonify({"users": sorted(users.values(), key=lambda u: u["name"])})


                @app.route("/add", methods=["POST"])
                def add_user():
                    data = request.get_json()
                    name = data["user"]
                    if name not in users:
                        create_user(name)
                    return jsonify(users[name])


                @app.route("/iou", methods=["POST"])
                def create_iou():
                    data = request.get_json()
                    lender = data["lender"]
                    borrower = data["borrower"]
                    amount = float(data["amount"])

                    updated = update_iou(lender, borrower, amount)
                    updated_sorted = sorted(updated, key=lambda u: u["name"])
                    return jsonify({"users": updated_sorted})


                if __name__ == "__main__":
                    app.run(debug=True)
====================================
⭐ Step 6 — Run the Flask API
In the terminal: " python app.py "

          You should see a message like: Running on http://127.0.0.1:5000
          This means your API is live.
====================================
⭐ Step 7 — Test the API (copy/paste these)
Open a new terminal window (not the one running Flask).

A. Add a user
        bash
        curl -X POST http://127.0.0.1:5000/add \
          -H "Content-Type: application/json" \
          -d "{\"user\": \"Adam\"}"
B.Add another user
        bash
        curl -X POST http://127.0.0.1:5000/add \
          -H "Content-Type: application/json" \
          -d "{\"user\": \"Bob\"}"
C.Create an IOU
        bash
        curl -X POST http://127.0.0.1:5000/iou \
          -H "Content-Type: application/json" \
          -d "{\"lender\": \"Adam\", \"borrower\": \"Bob\", \"amount\": 5.25}"
D.List all users
        bash
        curl -X GET http://127.0.0.1:5000/users
====================================
⭐ Step 8 — (Optional) Use VS Code Debugger
If you want to see the code execute line‑by‑line:

Open app.py

Click left of a line number to set a breakpoint

Go to Run → Start Debugging

Trigger a request (curl/Postman)

VS Code will pause at your breakpoint
→ you can inspect variables, step through code, etc.

This is the best way to learn how the API flows.

⭐ If you want, I can also help you with:
Setting up Postman to test the API

Adding SQLite persistence to the Flask version

Converting this into a multi‑file Flask project

Adding unit tests

Adding Swagger UI to Flask

====================================

ERROR while trying to execute- to test curl in powershell

1.   PS C:\Users\Jyoti\iou_flask_api> curl -X POST http://127.0.0.1:5000/add \
        >>   -H "Content-Type: application/json" \
        >>   -d "{\"user\": \"Adam\"}"
        >>

2.   Invoke-WebRequest : A parameter cannot be found that matches parameter name 'X'.
        At line:1 char:6
        + curl -X POST http://127.0.0.1:5000/add \
        +      ~~
            + CategoryInfo          : InvalidArgument: (:) [Invoke-WebRequest], ParameterB
          indingException
            + FullyQualifiedErrorId : NamedParameterNotFound,Microsoft.PowerShell.Commands
          .InvokeWebRequestCommand

        -H : The term '-H' is not recognized as the name of a cmdlet, function, script
        file, or operable program. Check the spelling of the name, or if a path was
        included, verify that the path is correct and try again.
        At line:2 char:3
        +   -H "Content-Type: application/json" \
        +   ~~
            + CategoryInfo          : ObjectNotFound: (-H:String) [], CommandNotFoundExcep
          tion
            + FullyQualifiedErrorId : CommandNotFoundException
 ====================================
SOLUTION->
====================================
this is a Windows PowerShell thing—not you doing anything wrong.

In PowerShell:  curl is actually an alias for Invoke-WebRequest

Unix-style flags like -X, -H, -d, and \ line continuation don’t work the same way

Let’s use PowerShell‑friendly commands.
====================================
Option 1 — Use Invoke-WebRequest (native PowerShell)
Run this in the same PowerShell window where your Flask app is running on http://127.0.0.1:5000:

powershell
            Invoke-WebRequest `
              -Uri "http://127.0.0.1:5000/add" `
              -Method POST `
              -Body '{"user": "Adam"}' `
              -ContentType "application/json"


To see the JSON response clearly:

          powershell
          (Invoke-WebRequest `
            -Uri "http://127.0.0.1:5000/add" `
            -Method POST `
            -Body '{"user": "Adam"}' `
            -ContentType "application/json").Content
====================================
Option 2 — Force real curl on Windows
If you have curl.exe installed, you can bypass the PowerShell alias:

powershell
curl.exe -X POST "http://127.0.0.1:5000/add" ^
  -H "Content-Type: application/json" ^
  -d "{\"user\":\"Adam\"}"
Note:

Use ^ for line continuation in PowerShell (not \)

Or just put it all on one line:

powershell
curl.exe -X POST "http://127.0.0.1:5000/add" -H "Content-Type: application/json" -d "{\"user\":\"Adam\"}"
Quick check: GET /users in PowerShell
powershell
(Invoke-WebRequest -Uri "http://127.0.0.1:5000/users" -Method GET).Content
====================================
Option1 worked . but option2 failed , after multiple trials also.
====================================

In [ ]:
## ✅ FastAPI Conversion of IOU API ->This version preserves the exact semantics of your original Flask implementation but upgrades:

🧱 Project Structure (simple version)
        iou_api/
        ├── main.py
        ├── models.py
        ├── schemas.py
        └── service.py

📌 Full FastAPI Implementation (copy‑paste ready)

schemas.py — Pydantic models
            from pydantic import BaseModel
            from typing import Dict, List

            class User(BaseModel):
                name: str
                owes: Dict[str, float] = {}
                owed_by: Dict[str, float] = {}
                balance: float = 0.0

            class AddUserRequest(BaseModel):
                user: str

            class IOURequest(BaseModel):
                lender: str
                borrower: str
                amount: float

            class UsersQuery(BaseModel):
                users: List[str]
======================
service.py — Core business logic

              users = {}

              def calculate_balance(user):
                  return sum(user["owed_by"].values()) - sum(user["owes"].values())

              def create_user(name):
                  user = {
                      "name": name,
                      "owes": {},
                      "owed_by": {},
                      "balance": 0.0
                  }
                  users[name] = user
                  return user


              def update_iou(lender, borrower, amount):
                  lender_user = users[lender]
                  borrower_user = users[borrower]

                  # Borrower owes lender
                  borrower_user["owes"][lender] = borrower_user["owes"].get(lender, 0) + amount
                  lender_user["owed_by"][borrower] = lender_user["owed_by"].get(borrower, 0) + amount

                  # Netting out cross-debts
                  if lender in borrower_user["owed_by"]:
                      cross = min(borrower_user["owed_by"][lender], borrower_user["owes"][lender])
                      borrower_user["owed_by"][lender] -= cross
                      borrower_user["owes"][lender] -= cross
                      if borrower_user["owed_by"][lender] == 0:
                          del borrower_user["owed_by"][lender]
                      if borrower_user["owes"][lender] == 0:
                          del borrower_user["owes"][lender]

                  if borrower in lender_user["owes"]:
                      cross = min(lender_user["owes"][borrower], lender_user["owed_by"][borrower])
                      lender_user["owed_by"][borrower] -= cross
                      lender_user["owes"][borrower] -= cross
                      if lender_user["owed_by"][borrower] == 0:
                          del lender_user["owed_by"][borrower]
                      if lender_user["owes"][borrower] == 0:
                          del lender_user["owes"][borrower]

                  # Recalculate balances
                  lender_user["balance"] = calculate_balance(lender_user)
                  borrower_user["balance"] = calculate_balance(borrower_user)

                  return [lender_user, borrower_user]
===========================================
main.py — FastAPI routes

            from fastapi import FastAPI, Body
            from typing import Optional

            from schemas import User, AddUserRequest, IOURequest, UsersQuery
            from service import users, create_user, update_iou

            app = FastAPI()


            @app.get("/users")
            def list_users(payload: Optional[UsersQuery] = Body(None)):
                if payload and payload.users:
                    selected = sorted([users[name] for name in payload.users], key=lambda u: u["name"])
                    return {"users": selected}
                return {"users": sorted(users.values(), key=lambda u: u["name"])}


            @app.post("/add")
            def add_user(req: AddUserRequest):
                user = create_user(req.user)
                return user


            @app.post("/iou")
            def create_iou(req: IOURequest):
                updated = update_iou(req.lender, req.borrower, req.amount)
                updated_sorted = sorted(updated, key=lambda u: u["name"])
                return {"users": updated_sorted}
=================================
🧪 Example Requests

Create user
        POST /add
        {"user": "Adam"}
Create IOU
        POST /iou
        {"lender": "Adam", "borrower": "Bob", "amount": 5.25}
List all users
        GET /users
List specific users
        GET /users
        {"users": ["Bob", "Adam"]}

In [ ]:
### IMPLEMENTATION -> step by step, end‑to‑end, inside VS Code.

1. Create the project in VS Code

A.Create a folder (e.g., iou_fastapi).
B.Open VS Code → File → Open Folder... → select iou_fastapi.
C.Inside that folder, create these files:

        main.py
        models.py
        database.py
        service.py
        schemas.py

2. Paste the code into the right files
models.py

            from sqlalchemy import Column, Integer, String, Float, ForeignKey
            from sqlalchemy.orm import relationship, declarative_base

            Base = declarative_base()

            class User(Base):
                __tablename__ = "users"

                id = Column(Integer, primary_key=True)
                name = Column(String, unique=True, nullable=False)

                lent = relationship("IOU", back_populates="lender", foreign_keys="IOU.lender_id")
                borrowed = relationship("IOU", back_populates="borrower", foreign_keys="IOU.borrower_id")


            class IOU(Base):
                __tablename__ = "ious"

                id = Column(Integer, primary_key=True)
                lender_id = Column(Integer, ForeignKey("users.id"))
                borrower_id = Column(Integer, ForeignKey("users.id"))
                amount = Column(Float, nullable=False)

                lender = relationship("User", foreign_keys=[lender_id], back_populates="lent")
                borrower = relationship("User", foreign_keys=[borrower_id], back_populates="borrowed")
===============================
database.py

                from sqlalchemy import create_engine
                from sqlalchemy.orm import sessionmaker

                DATABASE_URL = "sqlite:///./iou.db"

                engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
                SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)
=================================
service.py

              from sqlalchemy.orm import Session
              from models import User, IOU

              def get_user_summary(db: Session, user: User):
                  owes = {}
                  owed_by = {}

                  for iou in user.borrowed:
                      owes[iou.lender.name] = owes.get(iou.lender.name, 0) + iou.amount

                  for iou in user.lent:
                      owed_by[iou.borrower.name] = owed_by.get(iou.borrower.name, 0) + iou.amount

                  balance = sum(owed_by.values()) - sum(owes.values())

                  return {
                      "name": user.name,
                      "owes": owes,
                      "owed_by": owed_by,
                      "balance": balance
                  }


              def create_user(db: Session, name: str):
                  user = User(name=name)
                  db.add(user)
                  db.commit()
                  db.refresh(user)
                  return user


              def record_iou(db: Session, lender: str, borrower: str, amount: float):
                  lender_user = db.query(User).filter_by(name=lender).first()
                  borrower_user = db.query(User).filter_by(name=borrower).first()

                  iou = IOU(lender_id=lender_user.id, borrower_id=borrower_user.id, amount=amount)
                  db.add(iou)
                  db.commit()

                  return sorted(
                      [get_user_summary(db, lender_user), get_user_summary(db, borrower_user)],
                      key=lambda u: u["name"]
                  )
======================================
schemas.py

            from pydantic import BaseModel
            from typing import Dict, List


            class UserSchema(BaseModel):
                name: str
                owes: Dict[str, float] = {}
                owed_by: Dict[str, float] = {}
                balance: float = 0.0


            class AddUserRequest(BaseModel):
                user: str


            class IOURequest(BaseModel):
                lender: str
                borrower: str
                amount: float


            class UsersQuery(BaseModel):
                users: List[str]
=====================================
main.py

            from fastapi import FastAPI, Depends, Body
            from sqlalchemy.orm import Session

            from database import SessionLocal, engine
            from models import Base, User
            from service import create_user, record_iou, get_user_summary
            from schemas import AddUserRequest, IOURequest, UsersQuery

            Base.metadata.create_all(bind=engine)

            app = FastAPI()

            def get_db():
                db = SessionLocal()
                try:
                    yield db
                finally:
                    db.close()


            @app.get("/users")
            def list_users(payload: UsersQuery = Body(None), db: Session = Depends(get_db)):
                if payload and payload.users:
                    users = (
                        db.query(User)
                        .filter(User.name.in_(payload.users))
                        .order_by(User.name)
                        .all()
                    )
                else:
                    users = db.query(User).order_by(User.name).all()

                return {"users": [get_user_summary(db, u) for u in users]}


            @app.post("/add")
            def add_user(req: AddUserRequest, db: Session = Depends(get_db)):
                user = create_user(db, req.user)
                return get_user_summary(db, user)


            @app.post("/iou")
            def create_iou(req: IOURequest, db: Session = Depends(get_db)):
                updated = record_iou(db, req.lender, req.borrower, req.amount)
                return {"users": updated}
=======================
3. Create and activate a virtual environment

A.Open the VS Code terminal (`Ctrl+``).

B.From the project folder:

        bash
        python -m venv .venv
C.Activate it:

  Windows (PowerShell):

        bash
        .venv\Scripts\Activate
  macOS/Linux:

        bash
        source .venv/bin/activate
You should see (.venv) in the terminal prompt.
==================================
4. Install dependencies
In the same terminal:

        pip install fastapi uvicorn sqlalchemy
(Optionally: pip install "pydantic[email]" if needed, but base pydantic comes with FastAPI.)
================================
5. Run the FastAPI app with Uvicorn
From the project folder:

          bash
          uvicorn main:app --reload

main → filename main.py
app → FastAPI instance app = FastAPI()
--reload → auto‑reload on code changes

You should see something like: -> Uvicorn running on http://127.0.0.1:8000
===================================
6. Explore the API in the browser (Swagger UI)
Open your browser and go to: ->  http://127.0.0.1:8000/docs

You’ll see:

        /add (POST)
        /iou (POST)
        /users (GET)

============> You can try requests directly there:

A.Create a user
        Click /add → Try it out

        Request body:

        json
        {
          "user": "Adam"
        }
Execute → you’ll see the user JSON and iou.db file created in your folder.

B.Create another user
          json
          {
            "user": "Bob"
          }
Create an IOU
          /iou → Try it out:

          json
          {
            "lender": "Adam",
            "borrower": "Bob",
            "amount": 5.25
          }
You’ll see both users returned with updated owes, owed_by, and balance.

List users
/users → Execute with no body → all users.

          {
            "users": ["Bob", "Adam"]
          }
=========================================================
7. How the execution flows (mentally tracing it)
=========================================================
Pick /iou as an example:

          Uvicorn receives HTTP request → passes to FastAPI.

          FastAPI matches route /iou + method POST → calls create_iou.

          Request JSON is parsed into IOURequest (Pydantic model).

          db: Session = Depends(get_db) opens a DB session.

          record_iou:

          Loads lender and borrower from users table.

          Inserts a row into ious table.

          Re‑queries and aggregates IOUs to build owes, owed_by, balance.

          Result is returned as JSON to the client.

In [ ]:
Execution FLow->
┌───────────────────────────────┐                 ┌────────────────────────────────┐
│           FLASK API           │                 │           FASTAPI API          │
│        (WSGI synchronous)     │                 │        (ASGI async-ready)      │
└───────────────────────────────┘                 └────────────────────────────────┘

        Client Sends HTTP Request                         Client Sends HTTP Request
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │   WSGI Server (Gunicorn) │                 │   ASGI Server (Uvicorn)    │
        │   or Flask Dev Server    │                 │   or Hypercorn             │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Flask Routing Layer      │                 │ FastAPI Routing Layer      │
        │ Matches "/iou" endpoint  │                 │ Matches "/iou" endpoint    │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ request.get_json()       │                 │ Pydantic Model (IOURequest)│
        │ Manual JSON extraction   │                 │ Auto-validated input        │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Create DB Session        │                 │ get_db() Dependency        │
        │ db = SessionLocal()      │                 │ Auto-injected Session      │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Insert IOU Record        │                 │ Insert IOU Record          │
        │ db.add(...); commit()    │                 │ db.add(...); commit()      │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Query DB for totals      │                 │ Query DB for totals        │
        │ Manual SQLAlchemy calls  │                 │ Same SQLAlchemy calls      │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Compute Balance          │                 │ Compute Balance            │
        │ sum(lent) - sum(borrow)  │                 │ sum(lent) - sum(borrow)    │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ jsonify({...})           │                 │ Return dict (auto JSON)    │
        │ Manual serialization     │                 │ FastAPI auto-converts      │
        └──────────────────────────┘                 └────────────────────────────┘
                       │                                              │
                       ▼                                              ▼
        ┌──────────────────────────┐                 ┌────────────────────────────┐
        │ Response Returned        │                 │ Response Returned          │
        │ via WSGI                 │                 │ via ASGI                   │
        └──────────────────────────┘                 └────────────────────────────┘
=============================================================================================
┌───────────────────────────────┬────────────────────────────────┐
│           FLASK API           │           FASTAPI API          │
├───────────────────────────────┼────────────────────────────────┤
│ 1. HTTP Request Arrives       │ 1. HTTP Request Arrives        │
│    (WSGI server)              │    (ASGI server: Uvicorn)      │
│                               │                                │
│ 2. Flask Router Matches       │ 2. FastAPI Router Matches      │
│    → @app.post("/iou")        │    → @app.post("/iou")         │
│                               │                                │
│ 3. Flask Calls View Function  │ 3. FastAPI Calls Endpoint      │
│    → create_iou()             │    → create_iou(req, db)       │
│                               │                                │
│ 4. Request Body Handling      │ 4. Request Body Handling       │
│    request.get_json()         │    Pydantic model IOURequest   │
│    Manual parsing             │    Auto-validated fields        │
│                               │                                │
│ 5. DB Session Created         │ 5. DB Session Injected         │
│    db = SessionLocal()        │    db = Depends(get_db)        │
│    Manual lifecycle           │    Auto lifecycle (yield)       │
│                               │                                │
│ 6. Insert IOU Row             │ 6. Insert IOU Row              │
│    db.add(IOU(...))           │    db.add(IOU(...))            │
│    db.commit()                │    db.commit()                 │
│                               │                                │
│ 7. Query DB for Balances      │ 7. Query DB for Balances       │
│    total_lent = ...           │    total_lent = ...            │
│    total_borrowed = ...       │    total_borrowed = ...        │
│                               │                                │
│ 8. Compute Balance            │ 8. Compute Balance             │
│    sum(...) - sum(...)        │    sum(...) - sum(...)         │
│                               │                                │
│ 9. Build Response Dict        │ 9. Build Response Dict         │
│    {"name":..., "balance":...}│    {"name":..., "balance":...} │
│                               │                                │
│10. Serialize to JSON          │10. Auto JSON Serialization     │
│    jsonify(...)               │    FastAPI handles it          │
│                               │                                │
│11. Return HTTP Response       │11. Return HTTP Response        │
│    Flask sends JSON           │    FastAPI sends JSON          │
└───────────────────────────────┴────────────────────────────────┘

🧠 Key Differences (Conceptual Flow)

🔸 Flask
            Manual JSON parsing

            Manual DB session creation

            Manual validation

            Manual response serialization

            WSGI (sync only)

🔹 FastAPI
          Automatic JSON → Pydantic model

          Automatic validation

          Automatic DB session lifecycle

          Automatic JSON serialization

          ASGI (async‑ready)

🎯 Mental Model Summary

          Think of Flask as: “You do everything yourself.”

          Think of FastAPI as: “The framework does the plumbing; you focus on logic.”


In [ ]:
###side‑by‑side comparison of Flask and FastAPI IOU endpoint

        Imports
        App initialization
        DB session handling
        Route logic

🟦 Side‑by‑Side Comparison (Flask vs FastAPI)

1️⃣ Imports

Flask->python
        from flask import Flask, request, jsonify
        from database import SessionLocal, engine
        from models import Base, User, IOU

FastAPI->
        from fastapi import FastAPI, Depends
        from sqlalchemy.orm import Session
        from database import SessionLocal, engine
        from models import Base, IOU
        from schemas import IOURequest

Key difference:
FastAPI uses Depends + Pydantic models (IOURequest).
====================================================
2️⃣ App Initialization
Flask
      app = Flask(__name__)
      Base.metadata.create_all(bind=engine)

FastAPI->python
      app = FastAPI()
      Base.metadata.create_all(bind=engine)

Both create the DB tables the same way.
====================================================
3️⃣ Database Session Handling
      Flask	->python
            db = SessionLocal()

      FastAPI->python
            def get_db():
              db = SessionLocal()
            try:
              yield db
            finally:
              db.close()

FastAPI uses dependency injection, which is cleaner and safer.
====================================================
4️⃣ Route Definition
🔹 Route Declaration

      Flask->python
            @app.post("/iou")
            def create_iou():

      fastapi->python
            @app.post("/iou")
            def create_iou(req: IOURequest, db: Session = Depends(get_db)):


FastAPI automatically parses JSON into req.
====================================================
5️⃣ Request Body Parsing
Flask->python
    data = request.get_json()
    lender = data["lender"]
    borrower = data["borrower"]
    amount = data["amount"]

FastApi->python
    lender=req.lender
    borrower=req.borrower
    amount=req.amount


FastAPI gives you validated fields directly.
====================================================
6️⃣ Insert IOU Into DB
Flask->python
      db.add(IOU(lender=lender, borrower=borrower, amount=amount))
      db.commit()

FastApi->python
      db.add(IOU(lender=req.lender, borrower=req.borrower, amount=req.amount))
      db.commit()


Identical logic.
====================================================
7️⃣ Compute Balance
Flask->python
      total_lent = db.query(IOU).filter(IOU.lender == lender).all()
      total_borrowed = db.query(IOU).filter(IOU.borrower == lender).all()
      balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)

FastApi->python
      total_lent = db.query(IOU).filter(IOU.lender == req.lender).all()
      total_borrowed = db.query(IOU).filter(IOU.borrower == req.lender).all()
      balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)


Same logic — only variable names differ.
====================================================
8️⃣ Return Response
Flask->python
      return jsonify({ "name": lender, "balance": balance })

FastApi->python
      return { "name": req.lender, "balance": balance }


FastAPI auto‑serializes to JSON.
====================================================
✅ Final Side‑by‑Side (Full Code Blocks)
====================================================
🟧 Flask Version
====================================================
python
                          from flask import Flask, request, jsonify
                          from database import SessionLocal, engine
                          from models import Base, User, IOU

                          app = Flask(__name__)
                          Base.metadata.create_all(bind=engine)

                          @app.post("/iou")
                          def create_iou():
                              data = request.get_json()
                              lender = data["lender"]
                              borrower = data["borrower"]
                              amount = data["amount"]

                              db = SessionLocal()

                              db.add(IOU(lender=lender, borrower=borrower, amount=amount))
                              db.commit()

                              total_lent = db.query(IOU).filter(IOU.lender == lender).all()
                              total_borrowed = db.query(IOU).filter(IOU.borrower == lender).all()

                              balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)

                              return jsonify({
                                  "name": lender,
                                  "balance": balance
                              })

                          if __name__ == "__main__":
                              app.run(debug=True)
====================================================
🟩 FastAPI Version
====================================================
python
                      from fastapi import FastAPI, Depends
                      from sqlalchemy.orm import Session
                      from database import SessionLocal, engine
                      from models import Base, IOU
                      from schemas import IOURequest

                      app = FastAPI()
                      Base.metadata.create_all(bind=engine)

                      def get_db():
                          db = SessionLocal()
                          try:
                              yield db
                          finally:
                              db.close()

                      @app.post("/iou")
                      def create_iou(req: IOURequest, db: Session = Depends(get_db)):
                          db.add(IOU(lender=req.lender, borrower=req.borrower, amount=req.amount))
                          db.commit()

                          total_lent = db.query(IOU).filter(IOU.lender == req.lender).all()
                          total_borrowed = db.query(IOU).filter(IOU.borrower == req.lender).all()

                          balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)

                          return {
                              "name": req.lender,
                              "balance": balance
                          }
🎯 Quick Mental Model (Why They Feel Different)
Flask =
        manual everything

        manual JSON parsing

        manual DB session

        manual validation

        manual response serialization

FastAPI =
        automatic everything

        automatic JSON → Pydantic model

        automatic validation

        automatic DB session lifecycle

        automatic JSON response

In [ ]:
#####1️⃣ Project Folder Structure (Side‑by‑Side)
🟧 Flask
Code
        flask_iou_app/
        │── app.py
        │── models.py
        │── database.py
        │── requirements.txt
🟦 FastAPI
Code
        fastapi_iou_app/
        │── main.py
        │── models.py
        │── schemas.py
        │── database.py
        │── requirements.txt
Key difference:

FastAPI uses Pydantic schemas → Flask does not.
=======================================================
2️⃣ Install Dependencies
🟧 Flask -> pip install flask sqlalchemy
🟦 FastAPI -> pip install fastapi uvicorn sqlalchemy pydantic

3️⃣ Database Layer (Identical for Both)

database.py
        from sqlalchemy import create_engine
        from sqlalchemy.orm import sessionmaker, declarative_base

        SQLALCHEMY_DATABASE_URL = "sqlite:///./iou.db"

        engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
        SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)
        Base = declarative_base()
=========================================================
4️⃣ Models (Identical for Both)
models.py
            from sqlalchemy import Column, Integer, String, Float
            from database import Base

            class User(Base):
                __tablename__ = "users"
                id = Column(Integer, primary_key=True)
                name = Column(String, unique=True)

            class IOU(Base):
                __tablename__ = "ious"
                id = Column(Integer, primary_key=True)
                lender = Column(String)
                borrower = Column(String)
                amount = Column(Float)
===========================================================
5️⃣ Request Schema (FastAPI Only)
🟦 schemas.py

          from pydantic import BaseModel

          class IOURequest(BaseModel):
              lender: str
              borrower: str
              amount: float

Flask does not have this — you manually parse JSON.
===========================================================
6️⃣ API Code (Side‑by‑Side)
🟧 Flask Version — app.py
python
from flask import Flask, request, jsonify
from database import SessionLocal, engine
from models import Base, User, IOU

app = Flask(__name__)
Base.metadata.create_all(bind=engine)

@app.post("/iou")
def create_iou():
    data = request.get_json()
    lender = data["lender"]
    borrower = data["borrower"]
    amount = data["amount"]

    db = SessionLocal()

    db.add(IOU(lender=lender, borrower=borrower, amount=amount))
    db.commit()

    total_lent = db.query(IOU).filter(IOU.lender == lender).all()
    total_borrowed = db.query(IOU).filter(IOU.borrower == lender).all()

    balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)

    return jsonify({
        "name": lender,
        "balance": balance
    })

if __name__ == "__main__":
    app.run(debug=True)
===============================================================
🟦 FastAPI Version — main.py
python
from fastapi import FastAPI, Depends
from sqlalchemy.orm import Session
from database import SessionLocal, engine
from models import Base, IOU
from schemas import IOURequest

app = FastAPI()
Base.metadata.create_all(bind=engine)

def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

@app.post("/iou")
def create_iou(req: IOURequest, db: Session = Depends(get_db)):
    db.add(IOU(lender=req.lender, borrower=req.borrower, amount=req.amount))
    db.commit()

    total_lent = db.query(IOU).filter(IOU.lender == req.lender).all()
    total_borrowed = db.query(IOU).filter(IOU.borrower == req.lender).all()

    balance = sum(i.amount for i in total_lent) - sum(i.amount for i in total_borrowed)

    return {
        "name": req.lender,
        "balance": balance
    }
============================================================
============================================================
7️⃣ Running the App (Side‑by‑Side)
🟧 Flask ->
          "python app.py"
Runs at: http://127.0.0.1:5000/iou

🟦 FastAPI ->
      " uvicorn main:app --reload "

Runs at:  http://127.0.0.1:8000/iou
Auto‑docs: http://127.0.0.1:8000/docs
============================================================
============================================================
8️⃣ Execution Flow (Side‑by‑Side Mental Trace)
🟧 Flask Execution Flow
            Werkzeug receives HTTP request.
            Flask router matches /iou.
            request.get_json() parses JSON.
            You manually create DB session.
            Insert IOU row.
            Query totals.
            Manually return jsonify().
            Werkzeug sends response.

🟦 FastAPI Execution Flow
            Uvicorn receives HTTP request.
            FastAPI router matches /iou.
            JSON auto‑parsed into Pydantic model.
            Depends(get_db) opens DB session.
            Insert IOU row.
            Query totals.
            FastAPI auto‑serializes dict → JSON.
            Uvicorn sends response.

🎯 When to use which — the real decision guide
✅ Use Flask when:
          You want a simple, minimal API quickly.
          You’re building a small service, prototype, or internal tool.
          You want full control over every component.
          You’re migrating old Python apps (Flask has been around since 2010).
          You don’t need async performance.

Typical use cases:

          Small CRUD apps
          Simple ML model serving
          Internal dashboards
          Legacy systems

✅ Use FastAPI when:

          You want automatic validation using Pydantic.
          You need high performance or async (e.g., many concurrent requests).
          You want auto-generated Swagger docs.
          You’re building microservices, event-driven systems, or modern APIs.
          You want clean, typed, production-ready code with less boilerplate.

Typical use cases:

          High‑traffic APIs
          Microservices
          Real-time systems (async)
          ML model serving at scale
          Modern backend architectures

🧪 Code comparison (minimal API)
Flask example

          from flask import Flask, request, jsonify
          app = Flask(__name__)
          @app.route("/hello", methods=["GET"])
          def hello():
              return jsonify({"msg": "Hello from Flask"})

FastAPI example

          from fastapi import FastAPI
          app = FastAPI()
          @app.get("/hello")
          def hello():
              return {"msg": "Hello from FastAPI"}


FastAPI gives:
            automatic docs at /docs
            automatic validation
            async support

Flask gives:
            simplicity
            minimalism

🧩 When companies choose which
Companies choose Flask when:
            They want a simple, stable, predictable framework.
            They have existing Flask apps.
            They don’t need async or extreme performance.

Companies choose FastAPI when:
            They want modern Python typing.
            They want speed and scalability.
            They want built-in validation and documentation.
            They are building microservices.

🧠 Non‑obvious insight (the part most people miss)
FastAPI reduces bugs and onboarding time because:
          Request/response schemas are typed.
          Validation is automatic.
          Docs are auto-generated.

Flask gives more freedom, but that freedom means:
          You must enforce structure yourself.
          You must add validation manually.
          You must maintain consistency across teams.

For a single developer → Flask feels easier.
For a team or production system → FastAPI wins.